# Preparação da Base Analítica

## Projeto: Análise Descritiva do E-commerce Olist

## Contexto

Esta etapa corresponde à preparação dos dados para as análises estatísticas
descritivas do projeto.

A preparação é realizada a partir dos dados brutos do Brazilian E-Commerce
Public Dataset by Olist e utiliza como referência os achados obtidos durante
as auditorias financeira e temporal realizadas na etapa anterior.

O objetivo é transformar os dados disponíveis em bases analíticas estruturadas,
consistentes e adequadas para as análises posteriores, mantendo a rastreabilidade
das decisões tomadas durante o processo.

## 1. Objetivo da Preparação

O objetivo desta etapa é preparar os dados para a construção das bases
analíticas que serão utilizadas nas análises estatísticas descritivas.

A preparação buscará:

- definir o universo de cada análise;
- estabelecer o nível de observação de cada base;
- selecionar as tabelas e variáveis relevantes;
- integrar informações provenientes de diferentes tabelas;
- criar variáveis analíticas derivadas dos dados originais;
- aplicar regras de tratamento baseadas nos achados das auditorias;
- preservar os dados brutos e manter a rastreabilidade das transformações;
- construir bases analíticas adequadas às perguntas do projeto;
- validar a consistência estrutural e analítica das bases produzidas.

## 1.1 Perguntas Analíticas

A preparação dos dados deverá fornecer estrutura para responder,
principalmente, às seguintes questões:

### Comportamento dos pedidos

- Qual é o comportamento dos pedidos em termos de volume e características?

### Aspectos financeiros

- Como se distribuem os valores dos pedidos?
- Como se comportam os valores de frete?
- Como os valores de pagamento se distribuem?

### Desempenho logístico

- Quanto tempo os pedidos levam para serem processados e entregues?
- Como se distribuem os tempos das diferentes etapas da entrega?

### Atrasos

- Qual é a frequência de pedidos atrasados?
- Como se distribuem os atrasos?
- Como os pedidos atrasados se diferenciam dos pedidos entregues no prazo?

### Dimensões de análise

- Como os indicadores podem variar ao longo do tempo?
- Como os indicadores podem se comportar de acordo com características dos clientes e pedidos?

## 1.2 Relação com as Auditorias

As regras adotadas nesta etapa serão fundamentadas nos achados das auditorias
realizadas anteriormente.

A auditoria financeira avaliou a consistência das informações relacionadas
a pedidos, itens e pagamentos.

A auditoria temporal avaliou a consistência das informações relacionadas às
etapas de processamento, transporte e entrega dos pedidos.

Os resultados dessas auditorias serão utilizados para definir critérios de
seleção, validação, criação de indicadores e tratamento dos registros durante
a construção das bases analíticas.

Os dados brutos não serão alterados. As regras de preparação serão aplicadas
sobre cópias ou bases derivadas, preservando a rastreabilidade do processo.

# 2. Regras de Preparação Derivadas da Auditoria

## 2.1 Objetivo das Regras de Preparação

As regras de preparação foram definidas a partir dos problemas e padrões
identificados durante as auditorias financeira e temporal.

O objetivo não é alterar ou apagar os dados brutos, mas estabelecer critérios
explícitos para determinar quais registros e variáveis serão utilizados nas
diferentes bases analíticas.

Cada regra possui uma justificativa analítica e será aplicada somente quando
necessária para a análise correspondente.

## 2.2 Síntese dos Achados que Impactam a Preparação

A auditoria identificou os seguintes pontos relevantes para a preparação das
bases analíticas:

- existência de registros com inconsistências temporais;
- existência de registros com informações temporais incompletas;
- existência de valores extremos de atraso;
- existência de pedidos classificados como atrasados;
- necessidade de preservar a estrutura e os dados brutos;
- necessidade de evitar duplicação de pedidos durante a integração das tabelas;
- necessidade de definir critérios diferentes de tratamento conforme a análise
  realizada.

### Regra 1 — Inconsistência no tempo de transporte

Na auditoria temporal, foram identificados **1.350 registros com
`carrier_days < 0`** dentro do universo de pedidos entregues com informações
temporais completas. Esses valores indicam uma inconsistência cronológica entre
a aprovação do pedido e o encaminhamento à transportadora.

Durante a preparação, ao calcular `carrier_days` para todos os pedidos da base,
foram identificados **1.359 valores negativos**.

**Regra de preparação:** os valores negativos de `carrier_days` serão
considerados inválidos para cálculos que dependam de uma duração válida entre
a aprovação do pedido e o encaminhamento à transportadora.

Esses valores serão convertidos para `NaN` somente na variável derivada
`carrier_days`, preservando as demais informações dos pedidos e os dados
originais.

### Regra 2 — Inconsistência no tempo de entrega

Foram identificados **23 registros com `delivery_days < 0`**, indicando que a
data de entrega ao cliente antecede a data de encaminhamento à transportadora.

**Regra de preparação:** esses valores serão considerados inválidos para
cálculos que dependam de uma duração válida entre o encaminhamento e a entrega
ao cliente.

Os valores negativos serão convertidos para `NaN` somente na variável derivada
`delivery_days`, sem exclusão dos pedidos da base.

### Regra 3 — Informações temporais incompletas

Foram identificados registros em que uma ou mais datas necessárias para o
cálculo dos indicadores temporais estavam ausentes.

**Regra de preparação:** registros que não possuam todas as informações
necessárias para determinada métrica temporal não serão utilizados no cálculo
dessa métrica.

Não serão atribuídos valores artificiais às datas ausentes.

Os registros originais serão preservados, e a ausência de informação será
considerada somente na métrica afetada.

### Regra 4 — Valores extremos de atraso

Foram identificados **360 pedidos com `delay_days > 30`**, representando valores
extremos na distribuição dos atrasos.

Esses registros não serão automaticamente excluídos da análise, pois um valor
extremo não constitui, por si só, evidência de erro.

**Regra de preparação:** os valores extremos serão preservados e poderão ser
identificados posteriormente por meio do critério `delay_days > 30`, permitindo
análises específicas da cauda da distribuição e investigação de possíveis
padrões ou causas.

### Regra 5 — Pedidos atrasados

Os pedidos classificados como atrasados serão mantidos na base analítica.

A variável `delivery_late` será utilizada para diferenciar pedidos entregues
dentro do prazo daqueles entregues após a data estimada.

Essa classificação será utilizada posteriormente em análises descritivas e
comparativas relacionadas ao cumprimento do prazo de entrega.

### Regra 6 — Integração das informações financeiras

As tabelas `order_items` e `order_payments` possuem múltiplos registros
associados ao mesmo pedido.

Para preservar o grão de uma linha por pedido na base financeira, as
informações dessas tabelas serão previamente agregadas ao nível de `order_id`
antes da integração.

A integração será validada posteriormente para garantir que não ocorra
multiplicação indevida de registros ou distorção dos valores financeiros.


### Regra 7 — Preservação dos dados brutos


Os arquivos originais localizados no diretório `data/raw` não serão alterados.

Todas as transformações, filtros, agregações e variáveis derivadas serão
realizados em DataFrames derivados ou em bases analíticas específicas.

Essa abordagem preserva a rastreabilidade dos dados e permite reproduzir o
processo de preparação.

## 2.3 Matriz de Regras de Preparação

| Achado da auditoria | Impacto | Regra de preparação |
|---|---|---|
| `carrier_days < 0` | Inconsistência no tempo de transporte | Converter os valores inválidos para `NaN` e não utilizá-los em métricas que dependam de `carrier_days` válido |
| `delivery_days < 0` | Inconsistência no tempo de entrega | Converter os valores inválidos para `NaN` e não utilizá-los em métricas que dependam de `delivery_days` válido |
| Datas temporais necessárias ausentes | Intervalo não calculável | Não utilizar o registro na métrica temporal correspondente |
| `delay_days > 30` | Valor extremo | Manter e permitir identificação pelo critério; não excluir automaticamente |
| `delivery_late = True` | Pedido atrasado | Manter na análise e utilizar na classificação de cumprimento do prazo |
| Múltiplos itens/pagamentos por pedido | Risco de multiplicação de linhas | Agregar ao nível de `order_id` antes do merge |
| Dados brutos | Risco de perda de rastreabilidade | Preservar `data/raw` sem alterações |

# 3. Definição do Universo da Análise


## 3.1 Conceito de Universo Analítico

O universo analítico corresponde ao conjunto de registros elegíveis para
determinada análise, considerando as informações necessárias, os critérios
de qualidade dos dados e o objetivo analítico definido.

Um mesmo pedido poderá participar de diferentes análises e, dependendo da
análise realizada, poderá haver critérios específicos de inclusão ou exclusão.

A exclusão de um registro de determinado universo não implica sua exclusão
do projeto ou dos dados brutos. O registro será considerado inelegível apenas
para a análise cuja variável ou informação necessária esteja comprometida.

## 3.2 Universo de Pedidos

O universo de pedidos será utilizado para análises relacionadas às
características gerais dos pedidos presentes no dataset.

### Unidade de análise

1 linha = 1 pedido.

### Tabela principal

`orders`

### Identificador

`order_id`

### Critério geral

Um pedido poderá participar das análises gerais desde que possua as
informações necessárias para a variável específica analisada.

Problemas relacionados exclusivamente a informações temporais não implicam,
automaticamente, a exclusão do pedido deste universo.

## 3.3 Universo Financeiro

O universo financeiro será utilizado para análises relacionadas aos valores
dos pedidos, itens, fretes e pagamentos.

### Unidade de análise

1 linha = 1 pedido.

### Tabelas utilizadas

- `orders`
- `order_items`
- `order_payments`

### Critério geral

A base financeira será estruturada no nível de `order_id`, com as informações
de `order_items` e `order_payments` previamente agregadas ao nível do pedido.

Pedidos sem informação financeira completa serão preservados na base, mas não
serão utilizados nas métricas que dependam diretamente da informação ausente.

Inconsistências exclusivamente temporais não implicam, por si só, a exclusão
do pedido das análises financeiras.


## 3.4 Universo Temporal

O universo temporal será utilizado para análises relacionadas à duração das
etapas do processo de pedido e entrega.

### Unidade de análise

1 linha = 1 pedido.

### Tabela principal

`orders`

### Critérios de elegibilidade

O universo temporal é composto pelos pedidos com `order_status = delivered`
que possuem as informações necessárias para a análise temporal:

- `order_purchase_timestamp`;
- `order_approved_at`;
- `order_delivered_carrier_date`;
- `order_delivered_customer_date`;
- `order_estimated_delivery_date`.

Após a aplicação desses critérios, foram obtidos **96.455 pedidos elegíveis**
para a análise temporal.

Dentro desse universo, as métricas temporais são calculadas somente quando
os timestamps necessários apresentam uma sequência cronológica válida.

Valores temporalmente inconsistentes são tratados apenas na variável derivada
afetada, não implicando a exclusão do pedido das demais análises.


## 3.5 Universo de Atrasos

O universo de atrasos será utilizado para analisar o comportamento dos pedidos
em relação ao cumprimento do prazo estimado de entrega.

### Unidade de análise

1 linha = 1 pedido pertencente ao universo temporal.

### Variáveis principais

- `delay_days`
- `delivery_late`

### Critério de elegibilidade

O universo de atrasos utiliza os pedidos pertencentes ao universo temporal,
pois a classificação de atraso depende da existência tanto da data efetiva de
entrega quanto da data estimada de entrega.

A variável `delivery_late` é definida a partir da relação entre `delay_days`
e zero:

- `delivery_late = True` quando `delay_days > 0`;
- `delivery_late = False` quando `delay_days <= 0`.

Na base temporal final, foram identificados:

- **88.630 pedidos não atrasados**;
- **7.825 pedidos atrasados**.

Pedidos classificados como atrasados serão mantidos na análise, pois o atraso
constitui uma informação analítica relevante e não representa, por si só, um
erro nos dados.

Valores extremos de atraso também não serão automaticamente excluídos.


## 3.6 Universo de Clientes

O universo de clientes será utilizado quando a pergunta analítica exigir uma
perspectiva centrada no comportamento dos clientes.

### Unidade de análise

1 linha = 1 cliente real.

### Tabela principal

`customers`

### Identificador

`customer_unique_id`

O campo `customer_id` identifica o registro de cliente associado a um pedido,
enquanto `customer_unique_id` permite identificar o mesmo cliente ao longo de
diferentes pedidos.

Na base `customers`, foram identificados:

- **99.441 registros de `customer_id`**;
- **96.096 clientes únicos por `customer_unique_id`**.

Portanto, análises de comportamento, recorrência, quantidade de pedidos e
gasto por cliente deverão utilizar `customer_unique_id` como unidade de
identificação do cliente.

Informações de pedidos e dados financeiros poderão ser posteriormente
agregados nesse nível para construção da base analítica de clientes.

## 3.7 Matriz de Universos Analíticos

| Universo | Unidade de análise | Identificador | Finalidade |
|---|---|---|---|
| Pedidos | 1 pedido | `order_id` | Características gerais dos pedidos |
| Financeiro | 1 pedido | `order_id` | Valores, fretes, pagamentos e reconciliação financeira |
| Temporal | 1 pedido | `order_id` | Duração das etapas de pedido e entrega |
| Atrasos | 1 pedido | `order_id` | Cumprimento do prazo estimado de entrega |
| Clientes | 1 cliente real | `customer_unique_id` | Comportamento, recorrência e métricas agregadas por cliente |

## 3.8 Carregamento das Tabelas

In [334]:
import pandas as pd
import numpy as np

In [335]:
orders = pd.read_csv(
    "../data/raw/olist_orders_dataset.csv"
)

order_items = pd.read_csv(
    "../data/raw/olist_order_items_dataset.csv"
)

customers = pd.read_csv(
    "../data/raw/olist_customers_dataset.csv"
)

order_payments = pd.read_csv(
    "../data/raw/olist_order_payments_dataset.csv"
)

In [336]:
print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("customers:", customers.shape)
print("order_payments:", order_payments.shape)

orders: (99441, 8)
order_items: (112650, 7)
customers: (99441, 5)
order_payments: (103886, 5)


In [337]:
print("order_id único em orders:", orders["order_id"].is_unique)
print("customer_id único em customers:", customers["customer_id"].is_unique)

order_id único em orders: True
customer_id único em customers: True


In [338]:
print("Pedidos em orders:", orders["order_id"].nunique())
print("Linhas em orders:", len(orders))

Pedidos em orders: 99441
Linhas em orders: 99441


In [339]:
print("Clientes únicos em customers:", customers["customer_id"].nunique())
print("Linhas em customers:", len(customers))

Clientes únicos em customers: 99441
Linhas em customers: 99441


In [340]:
print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("customers:", customers.shape)
print("order_payments:", order_payments.shape)

orders: (99441, 8)
order_items: (112650, 7)
customers: (99441, 5)
order_payments: (103886, 5)


In [341]:
print("order_id único em orders:", orders["order_id"].is_unique)
print("customer_id único em customers:", customers["customer_id"].is_unique)

order_id único em orders: True
customer_id único em customers: True


In [342]:
print("Pedidos em orders:", orders["order_id"].nunique())
print("Linhas em orders:", len(orders))

Pedidos em orders: 99441
Linhas em orders: 99441


In [343]:
print("Clientes únicos em customers:", customers["customer_id"].nunique())
print("Linhas em customers:", len(customers))

Clientes únicos em customers: 99441
Linhas em customers: 99441


In [344]:
print("Pedidos únicos em order_items:", order_items["order_id"].nunique())
print("Linhas em order_items:", len(order_items))

print("Pedidos únicos em order_payments:", order_payments["order_id"].nunique())
print("Linhas em order_payments:", len(order_payments))

Pedidos únicos em order_items: 98666
Linhas em order_items: 112650
Pedidos únicos em order_payments: 99440
Linhas em order_payments: 103886


In [345]:
items_per_order = order_items.groupby("order_id").size()

items_per_order.describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64

In [346]:
payments_per_order = order_payments.groupby("order_id").size()

payments_per_order.describe()

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64

In [347]:
orders_without_payment = orders[
    ~orders["order_id"].isin(order_payments["order_id"])
]

orders_without_payment[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp"
    ]
]

,order_id,customer_id,order_status,order_purchase_timestamp
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38


In [348]:
print(
    "Pedidos com mais de 1 item:",
    (items_per_order > 1).sum()
)

Pedidos com mais de 1 item: 9803


In [349]:
print(
    "Pedidos com mais de 1 pagamento:",
    (payments_per_order > 1).sum()
)

Pedidos com mais de 1 pagamento: 2961


In [350]:
orders_without_payment[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04 00:00:00


## 4.1 Definição do Grão das Tabelas

A análise da estrutura das tabelas confirmou diferentes níveis de observação
entre as fontes utilizadas.

O grão representa a unidade de observação de cada tabela, sendo fundamental
para evitar interpretações incorretas e multiplicação indevida de registros
durante a integração das bases.

- `orders`: 1 linha representa 1 pedido.
- `customers`: 1 linha representa 1 registro associado a um `customer_id`.
- `order_items`: 1 linha representa 1 item pertencente a um pedido.
- `order_payments`: 1 linha representa 1 registro de pagamento associado a um
  pedido.

Foram identificados **9.803 pedidos com mais de um item** e **2.961 pedidos
com mais de um registro de pagamento**, confirmando relações de um-para-muitos
entre `orders` e essas tabelas.

Também foi identificado **1 pedido presente em `orders` sem registro
correspondente em `order_payments`**. Esse pedido será preservado na base,
mas sua ausência de informação financeira será considerada nas análises que
dependem diretamente dos registros de pagamento.

### 4.2 Grão das Bases Analíticas

Para as bases analíticas que utilizam o pedido como unidade de observação,
o grão definido será:

**1 linha = 1 pedido (`order_id`).**

Por esse motivo, as informações de `order_items` e `order_payments`, que
possuem múltiplos registros por pedido, serão previamente agregadas ao nível
de `order_id` antes da integração com `orders`.

Na análise de clientes, a unidade de observação será o cliente identificado
por `customer_unique_id`, permitindo consolidar diferentes pedidos
pertencentes ao mesmo cliente real.

Essa definição de grão será utilizada como critério de validação das bases
analíticas, garantindo que a integração das tabelas não provoque duplicação
ou multiplicação indevida de pedidos ou clientes.

In [351]:
print("ORDERS")
print(orders.columns.tolist())

print("\nORDER_ITEMS")
print(order_items.columns.tolist())

print("\nORDER_PAYMENTS")
print(order_payments.columns.tolist())

print("\nCUSTOMERS")
print(customers.columns.tolist())

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

ORDER_ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

ORDER_PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [352]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [353]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [354]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

In [355]:
for col in date_columns:
    print(col, orders[col].isna().sum())

order_purchase_timestamp 0
order_approved_at 160
order_delivered_carrier_date 1783
order_delivered_customer_date 2965
order_estimated_delivery_date 0


In [356]:
orders_prep = orders.copy()

In [357]:
for col in date_columns:
    orders_prep[col] = pd.to_datetime(orders_prep[col])

In [358]:
orders_prep.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [359]:
for col in date_columns:
    print(col, orders_prep[col].isna().sum())

order_purchase_timestamp 0
order_approved_at 160
order_delivered_carrier_date 1783
order_delivered_customer_date 2965
order_estimated_delivery_date 0


In [360]:
orders_prep[date_columns].head()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [361]:
orders_prep["approval_days"] = (
    orders_prep["order_approved_at"]
    - orders_prep["order_purchase_timestamp"]
).dt.total_seconds() / 86400

In [362]:
orders_prep["approval_days"].describe()

count    99281.000000
mean         0.434129
std          1.084917
min          0.000000
25%          0.008958
50%          0.014306
75%          0.607535
max        187.882523
Name: approval_days, dtype: float64

In [363]:
print("Valores ausentes:", orders_prep["approval_days"].isna().sum())

Valores ausentes: 160


In [364]:
negative_approval = orders_prep[
    orders_prep["approval_days"] < 0
]

print("Pedidos com approval_days negativo:", len(negative_approval))

Pedidos com approval_days negativo: 0


In [365]:
orders_prep.loc[
    orders_prep["approval_days"].idxmax(),
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "approval_days"
    ]
]

order_id                    1612081119e8f23745698ad3367cc14b
order_status                                     unavailable
order_purchase_timestamp                 2016-10-05 18:06:48
order_approved_at                        2017-04-11 15:17:38
approval_days                                     187.882523
Name: 47552, dtype: object

In [366]:
orders_prep.nlargest(
    10,
    "approval_days"
)[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "approval_days"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_approved_at,approval_days
47552,1612081119e8f23745698ad3367cc14b,unavailable,2016-10-05 18:06:48,2017-04-11 15:17:38,187.882523
62293,2e5dc86c8c4aa663549caf5e31de840d,processing,2017-02-03 00:04:49,2017-04-04 10:56:48,60.452766
4541,2e7a8482f6fb09756ca50c10d7bfc047,shipped,2016-09-04 21:15:19,2016-10-07 13:18:03,32.668565
4396,e5fa5a7210941f7d56d0208e4e071d35,canceled,2016-09-05 00:15:34,2016-10-07 13:17:15,32.542836
43697,0a5c74ccc786ced7903270de9d6c170a,unavailable,2018-01-18 23:14:36,2018-02-20 12:05:54,32.535625
96251,0a93b40850d3f4becf2f276666e01340,delivered,2018-01-20 14:24:50,2018-02-20 11:51:27,30.893484
55708,f7923db0430587601c2aef15ec4b8af4,delivered,2018-01-20 17:38:58,2018-02-20 12:05:54,30.768704
53475,490291524fddde2b31c2e6bec3d9e6da,canceled,2017-04-14 22:40:54,2017-05-13 02:45:06,28.169583
10071,809a282bbd5dbcabb6f2f724fca862ec,canceled,2016-09-13 15:24:19,2016-10-07 13:16:46,23.911424
83143,fdd647b689626410b725d1cce2ddf37c,processing,2017-12-04 10:09:35,2017-12-27 14:03:00,23.162095


In [367]:
orders_prep["carrier_days"] = (
    orders_prep["order_delivered_carrier_date"]
    - orders_prep["order_approved_at"]
).dt.total_seconds() / 86400

In [368]:
orders_prep["carrier_days"].describe()

count    97644.000000
mean         2.805038
std          3.549427
min       -171.219005
25%          0.875509
50%          1.818397
75%          3.580469
max        125.762569
Name: carrier_days, dtype: float64

In [369]:
print(
    "Valores ausentes:",
    orders_prep["carrier_days"].isna().sum()
)

Valores ausentes: 1797


In [370]:
negative_carrier = orders_prep[
    orders_prep["carrier_days"] < 0
]

print(
    "Pedidos com carrier_days negativo:",
    len(negative_carrier)
)

Pedidos com carrier_days negativo: 1359


In [371]:
print("Negativos encontrados agora:", len(negative_carrier))

print(
    negative_carrier[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "carrier_days"
        ]
    ].head(10)
)

Negativos encontrados agora: 1359
                             order_id order_purchase_timestamp  \
15   dcb36b511fcac050b97cd5c05de84dc3      2018-06-07 19:03:12   
64   688052146432ef8253587b930b01a06d      2018-04-22 08:48:13   
199  58d4c4747ee059eeeb865b349b41f53a      2018-07-21 12:49:32   
210  412fccb2b44a99b36714bca3fef8ad7b      2018-07-22 22:30:05   
415  56a4ac10a4a8f2ba7693523bb439eede      2018-07-22 13:04:47   
481  32e4fa9bb468884309b58b9348de70c3      2018-07-04 16:49:21   
483  4df92d82d79c3b52c7138679fa9b07fc      2018-07-24 11:32:11   
585  16e38caa92e342c7780f68832f832d4d      2018-05-07 01:09:09   
615  b9afddbdcfadc9a87b41a83271c3e888      2018-08-16 13:50:48   
817  6051e6d3da9a50b7325cbe9c81025062      2018-07-03 23:40:16   

      order_approved_at order_delivered_carrier_date  carrier_days  
15  2018-06-12 23:31:02          2018-06-11 14:54:00     -1.359051  
64  2018-04-24 18:25:22          2018-04-23 19:19:14     -0.962593  
199 2018-07-26 23:31:53         

In [372]:
percentual_negative_carrier = (
    len(negative_carrier) / orders_prep["carrier_days"].notna().sum()
) * 100

print(f"Percentual de carrier_days negativos: {percentual_negative_carrier:.2f}%")

Percentual de carrier_days negativos: 1.39%


In [373]:
delivered_orders_prep = orders_prep[
    orders_prep["order_status"] == "delivered"
].copy()

print("Pedidos entregues:", len(delivered_orders_prep))

Pedidos entregues: 96478


In [374]:
carrier_delivered = delivered_orders_prep[
    delivered_orders_prep["order_approved_at"].notna() &
    delivered_orders_prep["order_delivered_carrier_date"].notna()
].copy()

carrier_delivered["carrier_days"] = (
    carrier_delivered["order_delivered_carrier_date"]
    - carrier_delivered["order_approved_at"]
).dt.total_seconds() / 86400

negative_carrier_delivered = carrier_delivered[
    carrier_delivered["carrier_days"] < 0
]

print(
    "Pedidos entregues com carrier_days negativo:",
    len(negative_carrier_delivered)
)

Pedidos entregues com carrier_days negativo: 1350


In [375]:
orders_prep.loc[
    orders_prep["carrier_days"] < 0,
    "carrier_days"
] = np.nan

In [376]:
print(
    "Valores ausentes em carrier_days:",
    orders_prep["carrier_days"].isna().sum()
)

print(
    "Valores negativos em carrier_days:",
    (orders_prep["carrier_days"] < 0).sum()
)

Valores ausentes em carrier_days: 3156
Valores negativos em carrier_days: 0


# 5. Transformação dos Dados e Criação das Variáveis Analíticas

In [377]:
print("ORDERS")
print(orders.columns.tolist())

print("\nORDER_ITEMS")
print(order_items.columns.tolist())

print("\nORDER_PAYMENTS")
print(order_payments.columns.tolist())

print("\nCUSTOMERS")
print(customers.columns.tolist())

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

ORDER_ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

ORDER_PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [378]:
orders.info()
orders.head()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [379]:
for col in date_columns:
    print(col, orders[col].isna().sum())

order_purchase_timestamp 0
order_approved_at 160
order_delivered_carrier_date 1783
order_delivered_customer_date 2965
order_estimated_delivery_date 0


In [380]:
orders_prep = orders.copy()

In [381]:
for col in date_columns:
    orders_prep[col] = pd.to_datetime(orders_prep[col])

In [382]:
orders_prep.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [383]:
for col in date_columns:
    print(col, orders_prep[col].isna().sum())

order_purchase_timestamp 0
order_approved_at 160
order_delivered_carrier_date 1783
order_delivered_customer_date 2965
order_estimated_delivery_date 0


In [384]:
orders_prep[date_columns].head()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [385]:
orders_prep["approval_days"] = (
    orders_prep["order_approved_at"]
    - orders_prep["order_purchase_timestamp"]
).dt.total_seconds() / 86400

In [386]:
orders_prep["approval_days"].describe()

count    99281.000000
mean         0.434129
std          1.084917
min          0.000000
25%          0.008958
50%          0.014306
75%          0.607535
max        187.882523
Name: approval_days, dtype: float64

In [387]:
print("Valores ausentes:", orders_prep["approval_days"].isna().sum())

Valores ausentes: 160


In [388]:
negative_approval = orders_prep[
    orders_prep["approval_days"] < 0
]

print("Pedidos com approval_days negativo:", len(negative_approval))

Pedidos com approval_days negativo: 0


In [389]:
orders_prep.loc[
    orders_prep["approval_days"].idxmax(),
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "approval_days"
    ]
]

order_id                    1612081119e8f23745698ad3367cc14b
order_status                                     unavailable
order_purchase_timestamp                 2016-10-05 18:06:48
order_approved_at                        2017-04-11 15:17:38
approval_days                                     187.882523
Name: 47552, dtype: object

In [390]:
orders_prep.nlargest(
    10,
    "approval_days"
)[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "approval_days"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_approved_at,approval_days
47552,1612081119e8f23745698ad3367cc14b,unavailable,2016-10-05 18:06:48,2017-04-11 15:17:38,187.882523
62293,2e5dc86c8c4aa663549caf5e31de840d,processing,2017-02-03 00:04:49,2017-04-04 10:56:48,60.452766
4541,2e7a8482f6fb09756ca50c10d7bfc047,shipped,2016-09-04 21:15:19,2016-10-07 13:18:03,32.668565
4396,e5fa5a7210941f7d56d0208e4e071d35,canceled,2016-09-05 00:15:34,2016-10-07 13:17:15,32.542836
43697,0a5c74ccc786ced7903270de9d6c170a,unavailable,2018-01-18 23:14:36,2018-02-20 12:05:54,32.535625
96251,0a93b40850d3f4becf2f276666e01340,delivered,2018-01-20 14:24:50,2018-02-20 11:51:27,30.893484
55708,f7923db0430587601c2aef15ec4b8af4,delivered,2018-01-20 17:38:58,2018-02-20 12:05:54,30.768704
53475,490291524fddde2b31c2e6bec3d9e6da,canceled,2017-04-14 22:40:54,2017-05-13 02:45:06,28.169583
10071,809a282bbd5dbcabb6f2f724fca862ec,canceled,2016-09-13 15:24:19,2016-10-07 13:16:46,23.911424
83143,fdd647b689626410b725d1cce2ddf37c,processing,2017-12-04 10:09:35,2017-12-27 14:03:00,23.162095


In [391]:
orders_prep["carrier_days"] = (
    orders_prep["order_delivered_carrier_date"]
    - orders_prep["order_approved_at"]
).dt.total_seconds() / 86400

In [392]:
orders_prep["carrier_days"].describe()

count    97644.000000
mean         2.805038
std          3.549427
min       -171.219005
25%          0.875509
50%          1.818397
75%          3.580469
max        125.762569
Name: carrier_days, dtype: float64

In [393]:
print(
    "Valores ausentes:",
    orders_prep["carrier_days"].isna().sum()
)

Valores ausentes: 1797


# 6. Tratamento e Validação das Variáveis Temporais





## 6.1 Tratamento de inconsistências em `carrier_days`

Durante a criação da variável `carrier_days`, foram identificados registros
com duração negativa entre a aprovação do pedido e a entrega à transportadora.

Valores negativos representam inconsistências cronológicas, pois indicam que
a entrega à transportadora teria ocorrido antes da aprovação do pedido.

Considerando todos os pedidos da base, foram identificados 1.359 registros
com `carrier_days < 0`. Esses valores foram considerados inválidos para
análises que dependem da duração entre aprovação e entrega à transportadora.

Os pedidos não foram excluídos da base. Apenas os valores inválidos da
variável derivada `carrier_days` foram convertidos para `NaN`, preservando
as demais informações dos pedidos.

Após o tratamento, não permanecem valores negativos em `carrier_days`.
O número de valores ausentes passou de 1.797 para 3.156, correspondendo aos
1.797 registros originalmente sem informação suficiente para o cálculo e
aos 1.359 registros cujo intervalo calculado apresentou inconsistência
cronológica.

A decisão preserva a rastreabilidade dos dados e restringe a exclusão
apenas da métrica afetada pela inconsistência.

In [394]:
negative_carrier = orders_prep[
    orders_prep["carrier_days"] < 0
]

print(
    "Pedidos com carrier_days negativo:",
    len(negative_carrier)
)

Pedidos com carrier_days negativo: 1359


In [395]:
print(
    negative_carrier[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "carrier_days"
        ]
    ].head(10)
)

                             order_id order_status order_purchase_timestamp  \
15   dcb36b511fcac050b97cd5c05de84dc3    delivered      2018-06-07 19:03:12   
64   688052146432ef8253587b930b01a06d    delivered      2018-04-22 08:48:13   
199  58d4c4747ee059eeeb865b349b41f53a    delivered      2018-07-21 12:49:32   
210  412fccb2b44a99b36714bca3fef8ad7b    delivered      2018-07-22 22:30:05   
415  56a4ac10a4a8f2ba7693523bb439eede    delivered      2018-07-22 13:04:47   
481  32e4fa9bb468884309b58b9348de70c3    delivered      2018-07-04 16:49:21   
483  4df92d82d79c3b52c7138679fa9b07fc    delivered      2018-07-24 11:32:11   
585  16e38caa92e342c7780f68832f832d4d    delivered      2018-05-07 01:09:09   
615  b9afddbdcfadc9a87b41a83271c3e888    delivered      2018-08-16 13:50:48   
817  6051e6d3da9a50b7325cbe9c81025062    delivered      2018-07-03 23:40:16   

      order_approved_at order_delivered_carrier_date  carrier_days  
15  2018-06-12 23:31:02          2018-06-11 14:54:00     -1.3

In [396]:
percentual_negative_carrier = (
    len(negative_carrier)
    / orders_prep["carrier_days"].notna().sum()
) * 100

print(
    f"Percentual de carrier_days negativos: "
    f"{percentual_negative_carrier:.2f}%"
)

Percentual de carrier_days negativos: 1.39%


In [397]:
orders_prep.loc[
    orders_prep["carrier_days"] < 0,
    "carrier_days"
] = np.nan

In [398]:
print(
    "Valores ausentes após o tratamento:",
    orders_prep["carrier_days"].isna().sum()
)

print(
    "Valores negativos após o tratamento:",
    (orders_prep["carrier_days"] < 0).sum()
)

Valores ausentes após o tratamento: 3156
Valores negativos após o tratamento: 0


In [399]:
print(
    "Total de linhas em orders_prep:",
    len(orders_prep)
)

Total de linhas em orders_prep: 99441


## 6.2 Tratamento de inconsistências em `delivery_days`

A variável `delivery_days` representa o intervalo, em dias, entre o
encaminhamento do pedido à transportadora e a entrega ao cliente.

Valores negativos indicam uma inconsistência cronológica, pois significam
que a data de entrega ao cliente é anterior à data de encaminhamento à
transportadora.

Durante a preparação dos dados, foram identificados registros com
`delivery_days < 0`.

Esses valores serão considerados inválidos para análises que dependem de uma
duração válida entre o encaminhamento à transportadora e a entrega ao cliente.

Os pedidos não serão excluídos da base. Apenas os valores inválidos da
variável derivada `delivery_days` serão convertidos para `NaN`, preservando as
demais informações dos pedidos.

Após o tratamento, será realizada uma validação para confirmar a inexistência
de valores negativos e verificar o impacto do tratamento sobre os valores
ausentes.

In [400]:
orders_prep["delivery_days"] = (
    orders_prep["order_delivered_customer_date"]
    - orders_prep["order_delivered_carrier_date"]
).dt.total_seconds() / 86400

In [401]:
print(
    "delivery_days criada:",
    "delivery_days" in orders_prep.columns
)

delivery_days criada: True


In [402]:
negative_delivery = orders_prep[
    orders_prep["delivery_days"] < 0
]

print(
    "Pedidos com delivery_days negativo:",
    len(negative_delivery)
)

Pedidos com delivery_days negativo: 23


In [403]:
print(
    negative_delivery[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "delivery_days"
        ]
    ]
)

                               order_id order_status order_purchase_timestamp  \
6437   a1abeb653a4d4cd1e142ccb8c82cd069    delivered      2017-07-20 11:20:52   
9553   383aa8b2724fe452d9ccd9934a8c628b    delivered      2017-07-02 20:58:43   
13487  cb1134f9010d242e9515ad1c78ec0c39    delivered      2017-07-16 12:35:34   
14474  dceb62e8fa94b46006c9554fed743df0    delivered      2017-07-20 20:58:05   
19268  5f9d46795c3126674e52becb3a1a517f    delivered      2017-07-18 11:48:20   
21338  8c78d01de3a9009e23d6877a7cc9be20    delivered      2016-10-08 15:36:50   
22520  b27af682321527a6349f1761eb3f360c    delivered      2017-06-14 20:17:04   
25393  1cc3ae63caffff2d6c3ee3e78e074acf    delivered      2017-08-07 21:35:22   
25646  e37f11cae9985ca58f0b56f268720537    delivered      2017-07-26 11:46:34   
27470  fa3e37584f4fdb1ded0e0de700dfcb4e    delivered      2017-07-30 19:32:23   
34939  c1e2bf2b7dd3309f2f5356c6b63968fa    delivered      2017-02-10 10:19:10   
41636  b866af202be0692766081

In [404]:
orders_prep.loc[
    orders_prep["delivery_days"] < 0,
    "delivery_days"
] = np.nan

In [405]:
print(
    "Valores ausentes após o tratamento:",
    orders_prep["delivery_days"].isna().sum()
)

print(
    "Valores negativos após o tratamento:",
    (orders_prep["delivery_days"] < 0).sum()
)

Valores ausentes após o tratamento: 2989
Valores negativos após o tratamento: 0


In [406]:
print(
    "Quantidade de delivery_days negativos:",
    (orders_prep["delivery_days"] < 0).sum()
)

Quantidade de delivery_days negativos: 0


In [407]:
print(
    "Total de linhas em orders_prep:",
    len(orders_prep)
)

Total de linhas em orders_prep: 99441


## 6.2 Tratamento de inconsistências em `delivery_days`

A variável `delivery_days` representa o intervalo, em dias, entre o encaminhamento do pedido à transportadora e a entrega ao cliente.

Durante a preparação dos dados, foram identificados **23 registros com `delivery_days < 0`**. Esses valores representam inconsistências cronológicas, pois indicam que a entrega ao cliente teria ocorrido antes do encaminhamento do pedido à transportadora.

Os registros foram inspecionados individualmente e os valores negativos foram considerados inválidos para análises que dependem de uma duração válida entre essas duas etapas do processo de entrega.

Como não há evidências suficientes para determinar qual dos timestamps originais apresenta o problema, os dados de origem não foram alterados. Em vez disso, somente os valores inválidos da variável derivada `delivery_days` foram convertidos para `NaN`.

Após o tratamento, foram obtidos **2.989 valores ausentes** em `delivery_days`. Esse total corresponde aos **2.966 valores originalmente ausentes**, acrescidos dos **23 valores identificados como inconsistentes e convertidos para `NaN`**.

A validação confirmou a inexistência de valores negativos após o tratamento e a manutenção das **99.441 linhas** da base `orders_prep`.

Dessa forma, o tratamento preserva os pedidos e as informações originais, restringindo a exclusão apenas da métrica temporal afetada pela inconsistência.


## 6.3 Tratamento de inconsistências em `total_delivery_days`

## 6.3 Tratamento de inconsistências em `total_delivery_days`

A variável `total_delivery_days` representa o intervalo, em dias, entre a aprovação do pedido e a entrega ao cliente.

Essa variável permite avaliar o tempo total transcorrido entre essas duas etapas do processo de atendimento do pedido.

Valores negativos indicam uma inconsistência cronológica, pois significam que a data de entrega ao cliente é anterior à data de aprovação do pedido.

Durante a preparação dos dados, esses valores serão investigados antes da aplicação do tratamento.

Assim como nas demais variáveis temporais, os registros não serão excluídos da base. Caso sejam confirmadas inconsistências cronológicas, somente os valores inválidos da variável derivada `total_delivery_days` serão convertidos para `NaN`, preservando os dados originais e as demais informações dos pedidos.


In [408]:
orders_prep["total_delivery_days"] = (
    orders_prep["order_delivered_customer_date"]
    - orders_prep["order_approved_at"]
).dt.total_seconds() / 86400

In [409]:
print(
    "total_delivery_days criada:",
    "total_delivery_days" in orders_prep.columns
)

total_delivery_days criada: True


In [410]:
negative_total_delivery = orders_prep[
    orders_prep["total_delivery_days"] < 0
]

print(
    "Pedidos com total_delivery_days negativo:",
    len(negative_total_delivery)
)

Pedidos com total_delivery_days negativo: 61


In [411]:
print(
    negative_total_delivery[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "carrier_days",
            "delivery_days",
            "total_delivery_days"
        ]
    ]
)

                               order_id order_status order_purchase_timestamp  \
199    58d4c4747ee059eeeb865b349b41f53a    delivered      2018-07-21 12:49:32   
483    4df92d82d79c3b52c7138679fa9b07fc    delivered      2018-07-24 11:32:11   
1986   6e57e23ecac1ae881286657694444267    delivered      2018-08-09 17:36:47   
3659   f222c56f035b47dfa1e069a88235d730    delivered      2018-01-30 09:43:45   
11738  cf72398d0690f841271b695bbfda82d2    delivered      2017-09-01 18:45:33   
...                                 ...          ...                      ...   
89703  fcbf4f4ef049367f9f85af94ed3b6010    delivered      2018-04-20 15:11:15   
91801  4387477eec4b3c89b39f3f454940d059    delivered      2018-08-09 20:45:10   
93680  4f3a6e28d764cf896b1fceb0028422c8    delivered      2018-07-03 09:34:16   
94359  9c3186381b733d4304e2e416afc6bbc1    delivered      2018-07-28 20:49:05   
98359  5a41aefdf8010bbd69a5264f69213b73    delivered      2018-07-02 16:13:00   

        order_approved_at o

In [412]:
print(
    "Total de total_delivery_days negativos:",
    len(negative_total_delivery)
)

print(
    "Também possuem carrier_days negativo:",
    (
        negative_total_delivery["carrier_days"] < 0
    ).sum()
)

print(
    "Também possuem delivery_days negativo:",
    (
        negative_total_delivery["delivery_days"] < 0
    ).sum()
)

Total de total_delivery_days negativos: 61
Também possuem carrier_days negativo: 0
Também possuem delivery_days negativo: 0


In [413]:
customer_before_approval = negative_total_delivery[
    negative_total_delivery["order_delivered_customer_date"]
    < negative_total_delivery["order_approved_at"]
]

print(
    "Entrega ao cliente anterior à aprovação:",
    len(customer_before_approval)
)

Entrega ao cliente anterior à aprovação: 61


In [414]:
check_total = negative_total_delivery[
    [
        "carrier_days",
        "delivery_days",
        "total_delivery_days"
    ]
].copy()

check_total["soma_das_etapas"] = (
    check_total["carrier_days"]
    + check_total["delivery_days"]
)

print(check_total.head(10))

       carrier_days  delivery_days  total_delivery_days  soma_das_etapas
199             NaN       1.459248            -0.981644              NaN
483             NaN       1.173576            -2.190914              NaN
1986            NaN       3.138715            -2.965243              NaN
3659            NaN       1.015208            -3.139687              NaN
11738           NaN       6.751632            -2.326123              NaN
13470           NaN       2.094942            -2.269016              NaN
14562           NaN       4.293519            -5.078565              NaN
16345           NaN       1.155069            -2.215046              NaN
18097           NaN       1.396308            -0.748484              NaN
20557           NaN       2.050428            -0.143843              NaN


In [415]:
orders_prep.loc[
    orders_prep["total_delivery_days"] < 0,
    "total_delivery_days"
] = np.nan

In [416]:
print(
    "Valores negativos após o tratamento:",
    (orders_prep["total_delivery_days"] < 0).sum()
)

Valores negativos após o tratamento: 0


In [417]:
print(
    "Valores ausentes após o tratamento:",
    orders_prep["total_delivery_days"].isna().sum()
)

Valores ausentes após o tratamento: 3040


In [418]:
print(
    "Total de linhas em orders_prep:",
    len(orders_prep)
)

Total de linhas em orders_prep: 99441


In [419]:
orders_prep.loc[
    orders_prep["total_delivery_days"] < 0,
    "total_delivery_days"
] = np.nan

In [420]:
print(
    "Valores negativos após o tratamento:",
    (orders_prep["total_delivery_days"] < 0).sum()
)

print(
    "Valores ausentes após o tratamento:",
    orders_prep["total_delivery_days"].isna().sum()
)

print(
    "Total de linhas em orders_prep:",
    len(orders_prep)
)

Valores negativos após o tratamento: 0
Valores ausentes após o tratamento: 3040
Total de linhas em orders_prep: 99441


## 6.4 Tratamento e Validação de `delay_days`

## 6.4 Tratamento e Validação de `delay_days`

A variável `delay_days` representa a diferença, em dias, entre a data efetiva de entrega ao cliente e a data estimada de entrega.

Essa variável permite avaliar o cumprimento do prazo estimado de entrega.

Valores negativos representam entregas realizadas antes da data estimada, enquanto valores positivos representam entregas realizadas após a data estimada.

Valores iguais a zero representariam entregas realizadas exatamente na data estimada.

Durante a preparação, os valores extremos de `delay_days` serão investigados para verificar se apresentam evidências de inconsistência nos dados.

Valores extremos não serão automaticamente considerados inválidos, pois uma observação fora do padrão pode representar um evento real do processo de entrega.

In [421]:
orders_prep["delay_days"] = (
    orders_prep["order_delivered_customer_date"]
    - orders_prep["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

In [422]:
print(
    "delay_days criada:",
    "delay_days" in orders_prep.columns
)

delay_days criada: True


In [423]:
print(
    orders_prep["delay_days"].describe()
)

count    96476.000000
mean       -11.179120
std         10.186113
min       -146.016123
25%        -16.244384
50%        -11.948941
75%         -6.390000
max        188.975081
Name: delay_days, dtype: float64


In [424]:
print(
    "Valores ausentes:",
    orders_prep["delay_days"].isna().sum()
)

print(
    "Valores negativos:",
    (orders_prep["delay_days"] < 0).sum()
)

print(
    "Valores iguais a zero:",
    (orders_prep["delay_days"] == 0).sum()
)

print(
    "Valores positivos:",
    (orders_prep["delay_days"] > 0).sum()
)

Valores ausentes: 2965
Valores negativos: 88649
Valores iguais a zero: 0
Valores positivos: 7827


In [425]:
maior_atraso = orders_prep.loc[
    orders_prep["delay_days"].idxmax()
]

print(
    maior_atraso[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delay_days"
        ]
    ]
)

order_id                         1b3190b2dfa9d789e1f14c05b647a14a
order_status                                            delivered
order_purchase_timestamp                      2018-02-23 14:57:35
order_approved_at                             2018-02-23 15:16:14
order_delivered_carrier_date                  2018-02-26 18:49:07
order_delivered_customer_date                 2018-09-19 23:24:07
order_estimated_delivery_date                 2018-03-15 00:00:00
delay_days                                             188.975081
Name: 55619, dtype: object


In [426]:
top_atrasos = orders_prep.nlargest(
    10,
    "delay_days"
)

print(
    top_atrasos[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delay_days"
        ]
    ]
)

                               order_id order_status order_purchase_timestamp  \
55619  1b3190b2dfa9d789e1f14c05b647a14a    delivered      2018-02-23 14:57:35   
19590  ca07593549f1816d26a572e06dc1eab6    delivered      2017-02-21 23:31:27   
11399  47b40429ed8cce3aee9199792275433f    delivered      2018-01-03 09:44:01   
81401  2fe324febf907e3ea3f2aa9650869fa5    delivered      2017-03-13 20:17:10   
89130  285ab9426d6982034523a855f55a885e    delivered      2017-03-08 22:47:40   
61610  440d0d17af552815d15a9e41abe49359    delivered      2017-03-07 23:59:51   
68769  c27815f7e3dd0b926b58552628481575    delivered      2017-03-15 23:23:17   
40847  d24e8541128cea179a11a65176e0a96f    delivered      2017-06-12 13:14:11   
38509  0f4519c5f1c541ddec9f21b3bddd533a    delivered      2017-03-09 13:26:57   
54480  2d7561026d542c8dbd8f0daeadf67a43    delivered      2017-03-15 11:24:27   

      order_delivered_customer_date order_estimated_delivery_date  delay_days  
55619           2018-09-19 2

In [427]:
maior_antecipacao = orders_prep.loc[
    orders_prep["delay_days"].idxmin()
]

print(
    maior_antecipacao[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delay_days"
        ]
    ]
)

order_id                         0607f0efea4b566f1eb8f7d3c2397320
order_status                                            delivered
order_purchase_timestamp                      2018-03-06 09:47:07
order_approved_at                             2018-03-06 09:55:47
order_delivered_carrier_date                  2018-03-07 21:33:39
order_delivered_customer_date                 2018-03-09 23:36:47
order_estimated_delivery_date                 2018-08-03 00:00:00
delay_days                                            -146.016123
Name: 40094, dtype: object


In [428]:
top_antecipacoes = orders_prep.nsmallest(
    10,
    "delay_days"
)

print(
    top_antecipacoes[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delay_days"
        ]
    ]
)

                               order_id order_status order_purchase_timestamp  \
40094  0607f0efea4b566f1eb8f7d3c2397320    delivered      2018-03-06 09:47:07   
15791  c72727d29cde4cf870d569bf65edabfd    delivered      2017-02-07 18:01:15   
57160  eec7f369423b033e549c02f3c5381205    delivered      2018-02-06 20:44:56   
86444  c2bb89b5c1dd978d507284be78a04cb2    delivered      2017-05-23 22:28:36   
67488  40dc2ba6f322a17626aac6244332828c    delivered      2017-10-05 21:39:05   
79449  1a695d543b7302aa9446c8d5fbd632bf    delivered      2017-12-16 10:32:49   
62706  39e0115911bf404857e14baa7f097feb    delivered      2018-01-20 07:48:16   
95849  559eea5a72341a4c82dbce9884277cb7    delivered      2017-11-17 10:24:03   
63537  38930f76efb00b138f4d632e4d557341    delivered      2018-01-28 13:47:42   
37491  c5132855100a12d63ed4e8ae05f9594d    delivered      2017-10-13 14:45:03   

      order_delivered_customer_date order_estimated_delivery_date  delay_days  
40094           2018-03-09 2

In [429]:
orders_prep["delivery_late"] = (
    orders_prep["delay_days"] > 0
)

In [430]:
print(
    "Distribuição de delivery_late:"
)

print(
    orders_prep["delivery_late"].value_counts(dropna=False)
)

Distribuição de delivery_late:
delivery_late
False    91614
True      7827
Name: count, dtype: int64


In [431]:
orders_prep["delivery_late"] = (
    orders_prep["delay_days"] > 0
).astype("boolean")

orders_prep.loc[
    orders_prep["delay_days"].isna(),
    "delivery_late"
] = pd.NA

In [432]:
print(
    "Distribuição de delivery_late:"
)

print(
    orders_prep["delivery_late"].value_counts(dropna=False)
)

Distribuição de delivery_late:
delivery_late
False    88649
True      7827
<NA>      2965
Name: count, dtype: Int64


In [433]:
print(
    "True com delay_days <= 0:",
    (
        orders_prep.loc[
            orders_prep["delivery_late"] == True,
            "delay_days"
        ] <= 0
    ).sum()
)

print(
    "False com delay_days > 0:",
    (
        orders_prep.loc[
            orders_prep["delivery_late"] == False,
            "delay_days"
        ] > 0
    ).sum()
)

print(
    "delivery_late ausente com delay_days ausente:",
    (
        orders_prep.loc[
            orders_prep["delivery_late"].isna(),
            "delay_days"
        ].isna()
    ).sum()
)

True com delay_days <= 0: 0
False com delay_days > 0: 0
delivery_late ausente com delay_days ausente: 2965


## 6.4 Tratamento e Validação de `delay_days`

A variável `delay_days` representa a diferença, em dias, entre a data efetiva de entrega ao cliente e a data estimada de entrega.

Essa variável permite avaliar o cumprimento do prazo estimado de entrega.

Valores negativos representam entregas realizadas antes da data estimada, enquanto valores positivos representam entregas realizadas após a data estimada. Valores iguais a zero representariam entregas realizadas exatamente na data estimada.

Durante a preparação, foram identificados **96.476 registros com `delay_days` calculável** e **2.965 registros sem informação suficiente para o cálculo**.

A distribuição dos valores calculados apresentou:

* **88.649 valores negativos**, correspondentes a pedidos entregues antes da data estimada;
* **0 valores iguais a zero**;
* **7.827 valores positivos**, correspondentes a pedidos entregues após a data estimada.

Os valores extremos também foram investigados. O maior atraso identificado foi de aproximadamente **188,98 dias**, enquanto a maior antecipação foi de aproximadamente **146,02 dias**.

A inspeção dos registros extremos não apresentou evidência suficiente de inconsistência cronológica nos timestamps utilizados para o cálculo. Dessa forma, os valores extremos foram considerados observações potencialmente válidas e **não foram excluídos ou convertidos para `NaN`**.

Essa decisão evita a remoção automática de observações apenas por apresentarem valores fora do padrão, preservando informações que podem representar eventos reais do processo de entrega.

### Classificação de `delivery_late`

A variável `delivery_late` foi criada para identificar se o pedido foi entregue após a data estimada.

A classificação adotada foi:

* `True` → `delay_days > 0`, indicando entrega após o prazo estimado;
* `False` → `delay_days <= 0`, indicando entrega antecipada ou exatamente no prazo;
* `NaN` → `delay_days` ausente, indicando que não havia informações suficientes para determinar o cumprimento do prazo.

Após a classificação, foram identificados:

* **7.827 pedidos atrasados**;
* **88.649 pedidos não atrasados**;
* **2.965 pedidos sem classificação**, devido à ausência de informação necessária para o cálculo de `delay_days`.

A validação cruzada confirmou que não existem registros classificados como atrasados com `delay_days <= 0`, nem registros classificados como não atrasados com `delay_days > 0`. Também foi confirmado que os **2.965 registros sem classificação possuem `delay_days` ausente**.

Portanto, a variável `delivery_late` está consistente com a regra definida para `delay_days`.

Os valores extremos de atraso e antecipação permanecem preservados para análises posteriores, enquanto os registros sem informações suficientes permanecem sem classificação, evitando atribuir artificialmente uma situação de cumprimento ou descumprimento do prazo.


# 7. Construção e Validação da Base Temporal

## 7.1 Objetivo da Base Temporal

A base temporal tem como objetivo reunir os pedidos elegíveis para as análises
relacionadas ao tempo de processamento, transporte, entrega e cumprimento do
prazo estimado.

Essa base será construída a partir de `orders_prep`, que já contém as variáveis
temporais derivadas e os tratamentos definidos nas etapas anteriores.

O objetivo desta etapa não é realizar novos tratamentos nos dados, mas
construir uma base analítica consistente e validar se sua estrutura atende aos
critérios definidos para o universo temporal.

A base temporal terá como unidade de análise **1 linha = 1 pedido
(`order_id`)** e será composta pelos pedidos com status `delivered` que
possuem as informações temporais necessárias para a análise.

As inconsistências já identificadas durante a preparação das variáveis
temporais permanecerão representadas como valores ausentes (`NaN`) somente nas
variáveis derivadas afetadas, sem exclusão do pedido da base quando as demais
informações necessárias estiverem disponíveis.

## 7.2 Construção inicial do universo temporal

A base temporal será construída a partir dos pedidos com status `delivered`,
pois as análises de duração e cumprimento do prazo dependem da existência de
uma entrega ao cliente.

Inicialmente, será realizado apenas o filtro pelo status do pedido. Em seguida,
serão aplicados os critérios de completude das informações temporais necessárias.

In [434]:
temporal_base = orders_prep[
    orders_prep["order_status"] == "delivered"
].copy()

print("Pedidos entregues:", len(temporal_base))

Pedidos entregues: 96478


In [435]:
temporal_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

print(
    temporal_base[temporal_columns].isna().sum()
)

order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64


In [436]:
incomplete_temporal = temporal_base[
    temporal_base[temporal_columns].isna().any(axis=1)
].copy()

print(
    "Pedidos entregues com alguma informação temporal ausente:",
    len(incomplete_temporal)
)

Pedidos entregues com alguma informação temporal ausente: 23


In [437]:
print(
    incomplete_temporal[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ]
)

                               order_id order_status order_purchase_timestamp  \
3002   2d1e2d5bf4dc7227b3bfebb81328c15f    delivered      2017-11-28 17:44:07   
5323   e04abd8149ef81b95221e88f6ed9ab6a    delivered      2017-02-18 14:40:00   
16567  8a9adc69528e1001fc68dd0aaebbb54a    delivered      2017-02-18 12:45:31   
19031  7013bcfc1c97fe719a7b5e05e61c12db    delivered      2017-02-18 13:29:47   
20618  f5dd62b788049ad9fc0526e3ad11a097    delivered      2018-06-20 06:58:43   
22663  5cf925b116421afa85ee25e99b4c34fb    delivered      2017-02-18 16:48:35   
23156  12a95a3c06dbaec84bcfb0e2da5d228a    delivered      2017-02-17 13:05:55   
26800  c1d4211b3dae76144deccd6c74144a88    delivered      2017-01-19 12:48:08   
38290  d69e5d356402adc8cf17e08b5033acfb    delivered      2017-02-19 01:28:47   
39334  d77031d6a3c8a52f019764e68f211c69    delivered      2017-02-18 11:04:19   
43834  2ebdfc4f15f23b91474edf87475f108e    delivered      2018-07-01 17:05:11   
48401  7002a78c79c519ac54022

In [438]:
temporal_base = temporal_base.dropna(
    subset=temporal_columns
).copy()

print("Pedidos na base temporal:", len(temporal_base))

Pedidos na base temporal: 96455


## 7.3 Validação do Grão da Base Temporal

A base temporal possui como unidade de análise um pedido, representado pelo
identificador `order_id`.

Dessa forma, cada pedido deve aparecer uma única vez na base. Essa validação é
fundamental para garantir que não exista duplicação de pedidos e que as
estatísticas calculadas posteriormente não sejam distorcidas pela
multiplicação de registros.

In [439]:
print("Total de linhas:", len(temporal_base))
print("Total de order_id únicos:", temporal_base["order_id"].nunique())

Total de linhas: 96455
Total de order_id únicos: 96455


## 7.4 Validação de Duplicidades

Como o grão da base temporal é de uma linha por pedido, não deve existir mais
de um registro com o mesmo `order_id`.

A validação de duplicidades tem como objetivo confirmar que a construção da
base não provocou multiplicação de pedidos durante o processo de preparação.

A ausência de duplicidades garante que cada pedido contribua uma única vez para
as análises estatísticas realizadas posteriormente.

In [440]:
duplicated_orders = temporal_base["order_id"].duplicated().sum()

print("Pedidos duplicados:", duplicated_orders)

Pedidos duplicados: 0


## 7.5 Validação das Informações Temporais Obrigatórias

Após a aplicação do critério de completude, a base temporal deve possuir todas
as informações necessárias para definir o universo da análise.

A validação verifica se permanecem valores ausentes nas cinco variáveis
temporais utilizadas como critério de elegibilidade:

- `order_purchase_timestamp`;
- `order_approved_at`;
- `order_delivered_carrier_date`;
- `order_delivered_customer_date`;
- `order_estimated_delivery_date`.

A ausência de valores nessas variáveis poderia impedir o cálculo ou a
interpretação de determinadas métricas temporais. Portanto, a validação deve
confirmar que não existem valores ausentes nessas colunas na base temporal
final.

In [441]:
print(
    temporal_base[temporal_columns].isna().sum()
)

order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64


## 7.6 Validação das Variáveis Temporais Derivadas

Após a construção da base temporal, as variáveis derivadas criadas durante a
preparação devem ser novamente validadas no universo final da análise.

Essa validação tem como objetivo verificar a presença de valores ausentes e
confirmar que os tratamentos realizados anteriormente permanecem consistentes
na base temporal.

As variáveis analisadas são:

- `approval_days`;
- `carrier_days`;
- `delivery_days`;
- `total_delivery_days`;
- `delay_days`.

Valores ausentes podem permanecer nas variáveis derivadas quando a informação
necessária para seu cálculo estava ausente ou quando o valor calculado foi
considerado inválido durante o tratamento.

Esses valores não implicam, por si só, a exclusão do pedido da base temporal.

In [442]:
temporal_variables = [
    "approval_days",
    "carrier_days",
    "delivery_days",
    "total_delivery_days",
    "delay_days"
]

print(
    temporal_base[temporal_variables].isna().sum()
)

approval_days             0
carrier_days           1350
delivery_days            23
total_delivery_days      61
delay_days                0
dtype: int64


In [443]:
print("approval_days negativos:",
      (temporal_base["approval_days"] < 0).sum())

print("carrier_days negativos:",
      (temporal_base["carrier_days"] < 0).sum())

print("delivery_days negativos:",
      (temporal_base["delivery_days"] < 0).sum())

print("total_delivery_days negativos:",
      (temporal_base["total_delivery_days"] < 0).sum())

approval_days negativos: 0
carrier_days negativos: 0
delivery_days negativos: 0
total_delivery_days negativos: 0


### 7.6.3 Validação de `delay_days`

Diferentemente das demais variáveis temporais derivadas, valores negativos em
`delay_days` não representam inconsistências cronológicas.

A variável representa a diferença entre a data efetiva de entrega e a data
estimada de entrega. Portanto:

- valores negativos indicam entrega antecipada;
- valores positivos indicam entrega após o prazo estimado;
- valor igual a zero indica entrega exatamente na data estimada.

Dessa forma, a validação deve verificar a distribuição desses três grupos sem
considerar os valores negativos como inválidos.

In [444]:
print("delay_days negativos:",
      (temporal_base["delay_days"] < 0).sum())

print("delay_days iguais a zero:",
      (temporal_base["delay_days"] == 0).sum())

print("delay_days positivos:",
      (temporal_base["delay_days"] > 0).sum())

print("delay_days ausentes:",
      temporal_base["delay_days"].isna().sum())

delay_days negativos: 88630
delay_days iguais a zero: 0
delay_days positivos: 7825
delay_days ausentes: 0


## 7.7 Validação de `delivery_late`

A variável `delivery_late` classifica os pedidos de acordo com o cumprimento do
prazo estimado de entrega.

A regra definida durante a preparação é:

- `True` → `delay_days > 0`, indicando entrega após a data estimada;
- `False` → `delay_days <= 0`, indicando entrega antecipada ou realizada
  exatamente na data estimada;
- `NaN` → ausência de informação suficiente para determinar a classificação.

Como a base temporal final possui `delay_days` completo, espera-se que todos os
pedidos possuam uma classificação válida em `delivery_late`.

A validação também verificará se a distribuição da variável está de acordo com
os valores observados em `delay_days`.

In [445]:
print(
    temporal_base["delivery_late"].value_counts(dropna=False)
)

delivery_late
False    88630
True      7825
Name: count, dtype: Int64


In [446]:
print(
    "True com delay_days <= 0:",
    (
        temporal_base.loc[
            temporal_base["delivery_late"] == True,
            "delay_days"
        ] <= 0
    ).sum()
)

print(
    "False com delay_days > 0:",
    (
        temporal_base.loc[
            temporal_base["delivery_late"] == False,
            "delay_days"
        ] > 0
    ).sum()
)

print(
    "delivery_late ausente:",
    temporal_base["delivery_late"].isna().sum()
)

True com delay_days <= 0: 0
False com delay_days > 0: 0
delivery_late ausente: 0


## 7.8 Validação Final da Base Temporal

Após a construção e as validações individuais, a base temporal será submetida
a uma verificação final integrada.

O objetivo é confirmar que a base mantém:

- o universo de pedidos definido;
- o grão de uma linha por `order_id`;
- ausência de duplicidades;
- completude das variáveis temporais obrigatórias;
- consistência das variáveis temporais derivadas;
- classificação válida de `delivery_late`.

Essa etapa representa a validação estrutural final da base antes de sua
utilização nas análises estatísticas.

In [448]:
print("Quantidade de linhas:", len(temporal_base))
print("Quantidade de order_id únicos:", temporal_base["order_id"].nunique())
print("Pedidos duplicados:", temporal_base["order_id"].duplicated().sum())
print("Pedidos com status diferente de delivered:",
      (temporal_base["order_status"] != "delivered").sum())

Quantidade de linhas: 96455
Quantidade de order_id únicos: 96455
Pedidos duplicados: 0
Pedidos com status diferente de delivered: 0


In [449]:
print("Valores ausentes nas variáveis temporais obrigatórias:")
print(
    temporal_base[temporal_columns].isna().sum()
)

print("\nValores ausentes nas variáveis temporais derivadas:")
print(
    temporal_base[temporal_variables].isna().sum()
)

print("\nPedidos classificados em delivery_late:")
print(
    temporal_base["delivery_late"].value_counts(dropna=False)
)

Valores ausentes nas variáveis temporais obrigatórias:
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64

Valores ausentes nas variáveis temporais derivadas:
approval_days             0
carrier_days           1350
delivery_days            23
total_delivery_days      61
delay_days                0
dtype: int64

Pedidos classificados em delivery_late:
delivery_late
False    88630
True      7825
Name: count, dtype: Int64


## 7.9 Resumo Final da Base Temporal

Após a aplicação dos critérios definidos para o universo temporal e a
realização das validações estruturais e analíticas, a `temporal_base` foi
considerada apta para utilização nas análises temporais do projeto.

A base final possui:

- **96.455 pedidos**;
- **1 linha por `order_id`**;
- **nenhum `order_id` duplicado**;
- todos os pedidos com `order_status = delivered`;
- todas as cinco informações temporais obrigatórias preenchidas;
- variáveis temporais derivadas previamente tratadas e validadas;
- classificação `delivery_late` consistente com `delay_days`.

Os valores ausentes presentes em `carrier_days`, `delivery_days` e
`total_delivery_days` correspondem exclusivamente a situações em que a métrica
específica não pôde ser calculada de forma válida ou em que o valor derivado
foi considerado inconsistente durante a preparação.

Essas ausências não implicam a exclusão do pedido da base temporal, pois as
demais informações do registro permanecem válidas para outras análises.

A base `temporal_base` está, portanto, estruturada e validada para ser utilizada
nas análises descritivas relacionadas ao desempenho temporal dos pedidos e ao
cumprimento dos prazos de entrega.

In [450]:
print(temporal_base.info())

<class 'pandas.DataFrame'>
Index: 96455 entries, 0 to 99440
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96455 non-null  str           
 1   customer_id                    96455 non-null  str           
 2   order_status                   96455 non-null  str           
 3   order_purchase_timestamp       96455 non-null  datetime64[us]
 4   order_approved_at              96455 non-null  datetime64[us]
 5   order_delivered_carrier_date   96455 non-null  datetime64[us]
 6   order_delivered_customer_date  96455 non-null  datetime64[us]
 7   order_estimated_delivery_date  96455 non-null  datetime64[us]
 8   approval_days                  96455 non-null  float64       
 9   carrier_days                   95105 non-null  float64       
 10  delivery_days                  96432 non-null  float64       
 11  total_delivery_days            

# 8. Construção e Validação da Base Financeira

## 8.1 Objetivo da Base Financeira

A base financeira tem como objetivo reunir, no nível de pedido, as informações
necessárias para análises relacionadas aos valores dos itens, frete e
pagamentos.

A unidade de análise será o pedido, representado por `order_id`, de modo que
cada linha da base corresponda a um único pedido.

A base será construída a partir das tabelas `orders`, `order_items` e
`order_payments`.

As tabelas `order_items` e `order_payments`, que possuem múltiplos registros
para um mesmo pedido, serão previamente agregadas ao nível de `order_id`
antes da integração.

Pedidos sem informações financeiras completas não serão automaticamente
excluídos da base. Nesses casos, as variáveis cuja informação não puder ser
calculada permanecerão ausentes e serão consideradas apenas nas métricas que
dependam dessas informações.

A reconciliação entre o valor calculado a partir dos itens e frete e o valor
agregado dos pagamentos será realizada como etapa de validação da qualidade
financeira da base.

## 8.2 Agregação dos Valores dos Itens por Pedido

A tabela `order_items` possui múltiplos registros para um mesmo `order_id`,
pois cada linha representa um item pertencente ao pedido.

Para preservar o grão de uma linha por pedido na base financeira, os registros
serão agregados ao nível de `order_id`.

Serão calculadas:

- `total_price`: soma dos valores dos itens do pedido;
- `total_freight`: soma dos valores de frete;
- `total_order_value`: soma de `total_price` e `total_freight`.

Essa agregação transforma a informação do nível de item para o nível de pedido,
permitindo sua integração posterior com `orders` sem multiplicação de registros.

In [451]:
order_items_by_order = (
    order_items
    .groupby("order_id")
    .agg(
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum")
    )
    .reset_index()
)

In [452]:
order_items_by_order["total_order_value"] = (
    order_items_by_order["total_price"]
    + order_items_by_order["total_freight"]
)

In [453]:
print("Linhas:", len(order_items_by_order))
print(
    "order_id únicos:",
    order_items_by_order["order_id"].nunique()
)
print(
    "order_id único:",
    order_items_by_order["order_id"].is_unique
)

Linhas: 98666
order_id únicos: 98666
order_id único: True


## 8.3 Validação dos Valores Agregados dos Itens

Após a agregação dos registros de `order_items` ao nível de `order_id`, serão
validadas as variáveis financeiras derivadas para verificar sua completude e
coerência matemática.

A validação terá como objetivo confirmar:

- ausência de valores ausentes nas métricas agregadas;
- consistência entre `total_price` e `total_freight`;
- coerência de `total_order_value`;
- inexistência de valores negativos nas métricas financeiras.

In [454]:
print(
    order_items_by_order[
        [
            "total_price",
            "total_freight",
            "total_order_value"
        ]
    ].isna().sum()
)

total_price          0
total_freight        0
total_order_value    0
dtype: int64


In [455]:
financial_value_check = (
    order_items_by_order["total_price"]
    + order_items_by_order["total_freight"]
    - order_items_by_order["total_order_value"]
)

print(
    "Maior diferença absoluta:",
    financial_value_check.abs().max()
)

Maior diferença absoluta: 0.0


In [456]:
print(
    "total_price negativos:",
    (order_items_by_order["total_price"] < 0).sum()
)

print(
    "total_freight negativos:",
    (order_items_by_order["total_freight"] < 0).sum()
)

print(
    "total_order_value negativos:",
    (order_items_by_order["total_order_value"] < 0).sum()
)

total_price negativos: 0
total_freight negativos: 0
total_order_value negativos: 0


## 8.4 Agregação dos Pagamentos por Pedido

A tabela `order_payments` pode possuir mais de um registro de pagamento para o
mesmo pedido.

Para preservar o grão de uma linha por pedido na base financeira, os registros
de pagamento serão agregados ao nível de `order_id`.

Será calculado `total_payment_value`, correspondente à soma dos valores de
todos os pagamentos associados a cada pedido.

In [457]:
payments_by_order = (
    order_payments
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum")
    )
    .reset_index()
)

In [458]:
print("Linhas:", len(payments_by_order))
print(
    "order_id únicos:",
    payments_by_order["order_id"].nunique()
)
print(
    "order_id único:",
    payments_by_order["order_id"].is_unique
)

Linhas: 99440
order_id únicos: 99440
order_id único: True


## 8.5 Construção da Base Financeira no Nível de Pedido

Após a agregação das tabelas `order_items` e `order_payments`, as informações
financeiras serão integradas à tabela `orders`.

A tabela `orders` será utilizada como base principal para preservar todos os
pedidos presentes no dataset.

A integração das informações de itens será realizada por `order_id`, mantendo
uma linha por pedido.

Pedidos sem correspondência em `order_items` serão preservados e terão as
variáveis financeiras correspondentes como ausentes.

In [459]:
base_financeira = (
    orders[
        [
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp"
        ]
    ]
    .merge(
        order_items_by_order,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

In [460]:
print("Linhas:", len(base_financeira))
print(
    "order_id únicos:",
    base_financeira["order_id"].nunique()
)
print(
    "order_id único:",
    base_financeira["order_id"].is_unique
)

Linhas: 99441
order_id únicos: 99441
order_id único: True


In [461]:
print(
    "Pedidos sem informação de itens:",
    base_financeira["total_order_value"].isna().sum()
)

Pedidos sem informação de itens: 775


In [462]:
base_financeira = (
    base_financeira
    .merge(
        payments_by_order,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

In [463]:
print("Linhas:", len(base_financeira))
print(
    "order_id únicos:",
    base_financeira["order_id"].nunique()
)
print(
    "order_id único:",
    base_financeira["order_id"].is_unique
)

Linhas: 99441
order_id únicos: 99441
order_id único: True


In [464]:
print(
    "Pedidos sem informação de pagamento:",
    base_financeira["total_payment_value"].isna().sum()
)

Pedidos sem informação de pagamento: 1


## 8.6 Reconciliação Financeira

A reconciliação financeira compara o valor total registrado nos pagamentos com
o valor total calculado a partir dos itens e do frete de cada pedido.

A variável `payment_difference` representa a diferença entre o valor total pago
e o valor total do pedido:

`payment_difference = total_payment_value - total_order_value`

Valores próximos de zero indicam correspondência entre os dois valores,
enquanto diferenças superiores à tolerância definida serão investigadas como
divergências financeiras.

Pedidos sem informação suficiente para calcular a diferença permanecerão com
valor ausente nessa variável.

In [465]:
base_financeira["payment_difference"] = (
    base_financeira["total_payment_value"]
    - base_financeira["total_order_value"]
)

In [466]:
print(
    base_financeira["payment_difference"].describe()
)

count    98665.000000
mean         0.029092
std          1.129221
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: payment_difference, dtype: float64


In [467]:
base_financeira["payment_difference_rounded"] = (
    base_financeira["payment_difference"].round(2)
)

In [468]:
print(
    base_financeira["payment_difference_rounded"]
    .value_counts()
    .head(10)
)

payment_difference_rounded
 0.00    98089
-0.01      170
 0.01      103
 0.02       20
-0.02       16
 6.89        3
 0.03        3
-0.03        3
 1.55        2
 6.00        2
Name: count, dtype: int64


### Critério de Reconciliação

Para classificar a correspondência entre o valor do pedido e o valor dos
pagamentos, será utilizada uma tolerância operacional de **R$ 0,02**.

Pedidos com valor absoluto de `payment_difference_rounded` menor ou igual a
R$ 0,02 serão considerados reconciliados dentro da tolerância.

Diferenças superiores a esse limite serão classificadas como divergentes.

Quando não houver informação suficiente para calcular a diferença financeira,
o pedido será classificado separadamente como sem informação financeira
completa.

In [469]:
base_financeira["payment_reconciliation_status"] = np.select(
    [
        base_financeira["payment_difference_rounded"].isna(),
        base_financeira["payment_difference_rounded"].abs() <= 0.02,
    ],
    [
        "sem_informacao_financeira_completa",
        "reconciliado_dentro_tolerancia",
    ],
    default="divergente"
)

In [470]:
print(
    base_financeira["payment_reconciliation_status"]
    .value_counts()
)

payment_reconciliation_status
reconciliado_dentro_tolerancia        98398
sem_informacao_financeira_completa      776
divergente                              267
Name: count, dtype: int64


## 8.7 Investigação das Divergências Financeiras

Os pedidos classificados como `divergente` apresentaram diferença absoluta
superior à tolerância operacional definida na reconciliação.

Esses registros não serão automaticamente excluídos ou corrigidos, pois a
existência de uma diferença entre os valores não permite concluir, isoladamente,
que os dados estejam incorretos.

As divergências serão analisadas para identificar sua magnitude e possíveis
padrões, preservando os valores originais e a rastreabilidade da informação.

In [471]:
financial_divergences = base_financeira[
    base_financeira["payment_reconciliation_status"] == "divergente"
].copy()

print(
    "Quantidade de pedidos divergentes:",
    len(financial_divergences)
)

Quantidade de pedidos divergentes: 267


In [472]:
print(
    financial_divergences["payment_difference_rounded"].describe()
)

count    267.000000
mean      10.752734
std       18.900519
min      -51.620000
25%        2.655000
50%        6.330000
75%       13.540000
max      182.810000
Name: payment_difference_rounded, dtype: float64


## 8.7 Investigação das Divergências Financeiras

Os pedidos classificados como `divergente` apresentaram diferença absoluta
superior à tolerância operacional de R$ 0,02.

A existência de uma divergência não permite concluir, isoladamente, que os
dados estejam incorretos. Por esse motivo, os registros serão investigados
antes de qualquer decisão de tratamento.

A investigação buscará identificar a magnitude das diferenças e possíveis
padrões nos registros de pagamento associados aos pedidos divergentes.

In [473]:
print(
    financial_divergences[
        [
            "order_id",
            "total_order_value",
            "total_payment_value",
            "payment_difference_rounded"
        ]
    ]
    .sort_values(
        "payment_difference_rounded",
        key=abs,
        ascending=False
    )
    .head(20)
)

                               order_id  total_order_value  \
11791  ce6d150fb29ada17d2082f4847107665            1403.66   
48686  6e5fe7366a2e1bfbf3257dba0af1267f             287.91   
70865  70b742795bc441e94a44a084b6d9ce7a             466.93   
33150  996c7e73600ad3723e8627ab7bef81e4             587.90   
52985  70b7e94ea46d3e8b5bc12a50186edaf0             213.15   
85434  bc2c82b0ef78d2252b6176d1972db7c9             242.01   
94304  af9ffff2ce6b3defd34fd4c78857a379             413.17   
31661  262118ce178bb3e4590a3adcf6d62e6b             177.74   
68811  bfdb5bbb06458d600a33d61f5f287472             348.93   
8548   8d9c0dc8d5a2ce804f6b925d8f8e6c1d             254.45   
2615   b7579d24f5b2dd3e20f2e57d0e07d170             466.28   
30187  abf1130bc676c9dcadf91e24f5e30a30             459.36   
74562  b76190a2c095fba255ab46987545a660             227.91   
72098  32720c0c8b42f4c2cdb07cee5ec2444b             224.75   
33384  6b0ac1bdea322c7060eea92feb9e9a6f             327.87   
79025  e

In [474]:
divergent_payments = order_payments[
    order_payments["order_id"].isin(
        financial_divergences["order_id"]
    )
].copy()

In [475]:
print(
    "Registros de pagamento dos pedidos divergentes:",
    len(divergent_payments)
)

Registros de pagamento dos pedidos divergentes: 284


In [476]:
print(
    divergent_payments[
        [
            "order_id",
            "payment_sequential",
            "payment_type",
            "payment_installments",
            "payment_value"
        ]
    ].head(20)
)

                              order_id  payment_sequential payment_type  \
114   f090de1b2ed9f4e251662cb31e3c7127                   1  credit_card   
359   ce6d150fb29ada17d2082f4847107665                   1  credit_card   
845   6f393015477564624446a2b4c948c0f0                   1  credit_card   
1877  2240d9349c55949c40dbfcc98ece280c                   2      voucher   
2030  239f380355f65dcb68551f07d16fc4a8                   1  credit_card   
2155  164a6c920f4a6f51223792333af13e24                   1  credit_card   
2895  8092da256aefda13b330290d2ca86521                   1  credit_card   
2943  016726239765c18f66826453f39c64e3                   1  credit_card   
2968  e57d9d7eca079892fe0aee2b0cf96311                   1  credit_card   
2970  70b7e94ea46d3e8b5bc12a50186edaf0                   1  credit_card   
3038  a69aaa894f707814637e9794aa556f0f                   1  credit_card   
3508  c5bb8cf75ccd20ee373200ab31628770                   1  credit_card   
4294  29692492693482bc3ac

In [477]:
payment_frequency_divergent = (
    divergent_payments
    .groupby("order_id")
    .size()
)

print(
    payment_frequency_divergent.describe()
)

count    267.000000
mean       1.063670
std        0.244623
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        2.000000
dtype: float64


In [478]:
print(
    payment_frequency_divergent
    .value_counts()
    .sort_index()
)

1    250
2     17
Name: count, dtype: int64


In [479]:
print(
    divergent_payments["payment_type"].value_counts()
)

payment_type
credit_card    267
voucher          7
debit_card       7
boleto           3
Name: count, dtype: int64


In [480]:
print(
    divergent_payments.groupby("payment_type")["payment_value"].agg(
        ["count", "sum", "mean"]
    )
)

              count       sum        mean
payment_type                             
boleto            3   2720.48  906.826667
credit_card     267  32191.45  120.567228
debit_card        7   1165.51  166.501429
voucher           7    323.34   46.191429


In [481]:
print(
    order_payments["payment_type"].value_counts(normalize=True) * 100
)

payment_type
credit_card    73.922376
boleto         19.043952
voucher         5.558978
debit_card      1.471806
not_defined     0.002888
Name: proportion, dtype: float64


In [482]:
print(
    divergent_payments["payment_type"].value_counts(normalize=True) * 100
)

payment_type
credit_card    94.014085
voucher         2.464789
debit_card      2.464789
boleto          1.056338
Name: proportion, dtype: float64


### 8.7.6 Taxa de Divergência por Perfil de Pagamento

A análise anterior identificou concentração das divergências entre registros de
pagamento do tipo `credit_card`. Entretanto, a frequência absoluta não permite
determinar se esse método apresenta, de fato, maior incidência de divergências.

Para avaliar essa hipótese, será analisada a taxa de divergência no nível do
pedido, comparando pedidos que possuem pelo menos um pagamento com cartão de
crédito com pedidos que não possuem esse tipo de pagamento.

In [483]:
orders_with_credit_card = (
    order_payments
    .groupby("order_id")["payment_type"]
    .apply(lambda x: (x == "credit_card").any())
    .rename("has_credit_card")
    .reset_index()
)

In [484]:
base_financeira = (
    base_financeira
    .drop(columns=["has_credit_card"], errors="ignore")
    .merge(
        orders_with_credit_card,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

In [485]:
print(
    base_financeira.groupby("has_credit_card")[
        "payment_reconciliation_status"
    ].value_counts(normalize=True)
)

has_credit_card  payment_reconciliation_status     
False            reconciliado_dentro_tolerancia        0.988184
                 sem_informacao_financeira_completa    0.011380
                 divergente                            0.000436
True             reconciliado_dentro_tolerancia        0.989922
                 sem_informacao_financeira_completa    0.006719
                 divergente                            0.003359
Name: proportion, dtype: float64


## 8.8 Regra de Tratamento das Divergências Financeiras

A reconciliação financeira identificou 267 pedidos com divergências superiores
à tolerância operacional de R$ 0,02.

A investigação exploratória identificou maior concentração dessas divergências
entre pedidos que possuem pelo menos um pagamento por cartão de crédito.
Entretanto, essa associação não é suficiente para determinar a causa das
diferenças.

Como não foi identificada evidência suficiente para corrigir os valores
originais de forma confiável, os pedidos divergentes não serão excluídos nem
terão seus valores financeiros alterados.

As divergências permanecerão sinalizadas na variável
`payment_reconciliation_status`, preservando os valores originais e a
rastreabilidade dos dados.

Pedidos sem informação financeira completa também serão preservados, sendo
classificados separadamente para evitar que a ausência de informação seja
confundida com uma divergência financeira.

In [486]:
print("Linhas:", len(base_financeira))
print(
    "Pedidos únicos:",
    base_financeira["order_id"].nunique()
)
print(
    "Order_id único:",
    base_financeira["order_id"].is_unique
)
print(
    "Duplicados:",
    base_financeira["order_id"].duplicated().sum()
)

Linhas: 99441
Pedidos únicos: 99441
Order_id único: True
Duplicados: 0


In [488]:
financial_value_check = (
    base_financeira["total_price"]
    + base_financeira["total_freight"]
    - base_financeira["total_order_value"]
)

print(
    "Maior diferença na composição do valor do pedido:",
    financial_value_check.abs().max()
)

Maior diferença na composição do valor do pedido: 0.0


In [489]:
payment_check = (
    base_financeira["total_payment_value"]
    - base_financeira["total_order_value"]
    - base_financeira["payment_difference"]
)

print(
    "Maior diferença na fórmula de payment_difference:",
    payment_check.abs().max()
)

Maior diferença na fórmula de payment_difference: 0.0


## 8.9 Validação Final da Base Financeira

A etapa final de validação verifica os valores ausentes das principais
variáveis financeiras e confirma se a classificação de reconciliação preserva
a distinção entre pedidos reconciliados, divergentes e pedidos sem informação
financeira completa.

Os valores ausentes serão mantidos quando resultarem da ausência de registros
nas tabelas financeiras de origem, sem atribuição de valores artificiais.

In [490]:
print("Valores ausentes:")
print(
    base_financeira[
        [
            "total_price",
            "total_freight",
            "total_order_value",
            "total_payment_value",
            "payment_difference",
            "payment_difference_rounded",
            "payment_reconciliation_status"
        ]
    ].isna().sum()
)

Valores ausentes:
total_price                      775
total_freight                    775
total_order_value                775
total_payment_value                1
payment_difference               776
payment_difference_rounded       776
payment_reconciliation_status      0
dtype: int64


In [491]:
print("\nStatus de reconciliação:")
print(
    base_financeira["payment_reconciliation_status"]
    .value_counts(dropna=False)
)


Status de reconciliação:
payment_reconciliation_status
reconciliado_dentro_tolerancia        98398
sem_informacao_financeira_completa      776
divergente                              267
Name: count, dtype: int64


## 8.10 Resumo Final da Base Financeira

Após a agregação das informações de itens e pagamentos, integração com a
tabela `orders` e validação da reconciliação financeira, a base financeira
final foi considerada apta para utilização nas análises financeiras do projeto.

A base possui:

- **99.441 pedidos**;
- **1 linha por `order_id`**;
- **99.441 `order_id` únicos**;
- **0 pedidos duplicados**;
- preservação de todos os pedidos presentes em `orders`;
- valores de itens e frete agregados no nível do pedido;
- valores de pagamento agregados no nível do pedido;
- reconciliação financeira validada.

A reconciliação classificou os pedidos em três grupos:

- **98.398 pedidos reconciliados dentro da tolerância operacional de R$ 0,02**;
- **267 pedidos divergentes**;
- **776 pedidos sem informação financeira completa**.

Os 267 pedidos divergentes foram mantidos sem alteração dos valores originais,
pois não foi identificada evidência suficiente para determinar uma correção
confiável. As divergências permaneceram sinalizadas por
`payment_reconciliation_status`.

Os 776 pedidos sem informação financeira completa também foram preservados na
base. A ausência de informação não foi substituída artificialmente e será
considerada apenas nas métricas que dependem diretamente dos dados financeiros
ausentes.

Dessa forma, a base financeira mantém o grão de uma linha por pedido,
preserva a rastreabilidade dos dados de origem e está estruturada para as
análises descritivas financeiras posteriores.

In [492]:
print(base_financeira.info())

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order_id                       99441 non-null  str    
 1   customer_id                    99441 non-null  str    
 2   order_status                   99441 non-null  str    
 3   order_purchase_timestamp       99441 non-null  str    
 4   total_price                    98666 non-null  float64
 5   total_freight                  98666 non-null  float64
 6   total_order_value              98666 non-null  float64
 7   total_payment_value            99440 non-null  float64
 8   payment_difference             98665 non-null  float64
 9   payment_difference_rounded     98665 non-null  float64
 10  payment_reconciliation_status  99441 non-null  str    
 11  has_credit_card                99440 non-null  object 
dtypes: float64(6), object(1), str(5)
memory usage: 9.1+ MB
No

## 8.11 Finalização da Base Financeira

Após as validações estruturais e financeiras, a base será finalizada com a
seleção das variáveis necessárias para as análises financeiras.

A variável `has_credit_card` foi utilizada exclusivamente durante a investigação
exploratória das divergências financeiras e não integra a base financeira
final.

A variável `order_purchase_timestamp` será mantida como tipo temporal para
permitir análises financeiras ao longo do tempo.

In [493]:
base_financeira["order_purchase_timestamp"] = pd.to_datetime(
    base_financeira["order_purchase_timestamp"]
)

In [494]:
base_financeira = base_financeira.drop(
    columns=["has_credit_card"]
)

In [495]:
base_financeira_final = base_financeira[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "total_price",
        "total_freight",
        "total_order_value",
        "total_payment_value",
        "payment_difference",
        "payment_difference_rounded",
        "payment_reconciliation_status"
    ]
].copy()

In [497]:
print(base_financeira_final.info())

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   total_price                    98666 non-null  float64       
 5   total_freight                  98666 non-null  float64       
 6   total_order_value              98666 non-null  float64       
 7   total_payment_value            99440 non-null  float64       
 8   payment_difference             98665 non-null  float64       
 9   payment_difference_rounded     98665 non-null  float64       
 10  payment_reconciliation_status  99441 non-null  str           
dtypes: datetime64[us](1), floa

In [498]:
print("Linhas:", len(base_financeira_final))
print(
    "order_id únicos:",
    base_financeira_final["order_id"].nunique()
)
print(
    "Duplicados:",
    base_financeira_final["order_id"].duplicated().sum()
)

Linhas: 99441
order_id únicos: 99441
Duplicados: 0


##  Objetivo da preparação

A etapa de preparação dos dados tem como objetivo transformar os dados
auditados em bases analíticas adequadas para as etapas posteriores de
estatística descritiva e inferencial.

As transformações realizadas nesta etapa são orientadas pelos problemas,
inconsistências e características estruturais identificados durante a
auditoria dos dados.

Entre as principais atividades estão a conversão de tipos de dados, a
criação de variáveis analíticas, a definição dos universos de análise, o
tratamento de inconsistências, a agregação de tabelas com diferentes
níveis de granularidade e a construção das bases analíticas.

As decisões de preparação são aplicadas de forma documentada e
rastreável, preservando os dados brutos e evitando alterações
irreversíveis na fonte original.

Quando uma inconsistência afeta apenas uma variável ou métrica específica,
o tratamento é realizado de forma localizada, evitando a exclusão
indiscriminada de registros que permanecem úteis para outras análises.

Ao final desta etapa, espera-se obter bases com granularidade definida,
variáveis adequadas aos objetivos analíticos, regras de tratamento
documentadas e validações que confirmem a integridade das transformações
realizadas.
